## 2 Layer Neural Network MNIST Dataset
The goal was to build "from scratch" a neural network capable of recognizing and labeling number from the MNIST dataset accurately without the direct use of third party libraries. Disclosure numpy was used as a bench mark to ensure numerical function accuracy. Test set has temporarily been commented out at the moment.

### Model features
#### Images
Images were stripped into single long slices of (1,784) vectors for computational purposes. 

#### Weights
Two sets of weights first with the shape (784, hidden_value) and then the second with (hidden_value, 10) were used, hidden value was arbitraliy set to 100 in this instance. Random weight values were set between (-0.1, 0.1) and a random seed specified for reproducibility. 

#### Activation functions
ReLU was used as our activation function. It serves the purpose introducing non linearity to our function. relu2deriv is used to help assign "blame" during backpropagation. 

#### Dropout mask
A dropout mask was employed to combat memorization and protect generalization. We will see how well that went when we uncover tests later. 


### Notes
For perfomance reasons only an excerpt of the total MNIST data set was used. Improvements can be made to performance by employing our own vectorization library written in C. Will work on that. This will allow us to attempt even deeper models, example a nested convolutional model in search of better test results. Future versions may inlcude a validation group as well.  This will allow us to pursue improved results without fear of bias.

In [6]:
import random
import sys
from enum import Enum
from keras.datasets import mnist

class Operator(Enum):
    ADD = 1
    MINUS = 2
    DELTA = 3
    SQUARE = 4
    MULT = 5
    MMINUS = 6
    
    
def arrayTypeAssertion(array, name=None):
    """
    Asserts that input is of type sequence
    Input: array(iterable), name (string)
    Output: None
    """
    assert isinstance(array, (list,tuple)), f"{name.title() if name else "Array"} should be of type either list or tuple" 

def matrixShapeAssertion(*, a_shape, b_shape, transpose):
    """
    Asserts that two matrices have the right dimension for matrix multiplication
    Input: a_shape(iterable), b_shape(iterable), transpose (iterable)
    Output: Nothing/Failure Message
    """
    assert a_shape[1 - transpose[0]] == b_shape[0 - transpose[1]], "Matrices with the following dimensions are unsuited for this operation"

def matrixTypeAssertion(matrix, name=None):
    assert isinstance(matrix[0], (list, tuple)), f"{name.title() if name else "Matrix" } should be of type nested iterable" 
    
def scalarXarray(*, scalar, array):
    """
    Multiplies an array by a scalar (elementwise)
    Input: scalar (number), array (iterable)
    Output: elementwise scalar multiplication result (iterable)
    """
    arrayTypeAssertion(array)
    assert (type(scalar) is int) or (type(scalar) is float), "Scalar should be of type number"
    
    return [el * scalar for el in array]


def getShape(*, array, shape=None):
    """
    Returns the shape of an array
    Input: array (iterable), shape (iterable)
    Output: shape of array (iterable)
    """
    arrayTypeAssertion(array)
    if shape is None:
        shape = []
    shape.append(len(array))
    if array != [] and isinstance(array[0], (list, tuple)):
        return getShape(array=array[0], shape=shape)
    else:
        return(tuple(shape))
        

def scalarTensorDiv(*, array, scalar):
    """
    Elementwise division of tensor by scalar
    Input: array (iterable), scalar (number)
    Output: output (iterable)
    """
    arrayTypeAssertion(array)
    output = []
    for item in array:
        if isinstance(item, (list, tuple)):
           output.append(scalarTensorDiv(array=item, scalar=scalar))
        else: 
            output.append(item / scalar)
    return output

def scalarTensorMul(*, array, scalar):
    """
    Returns scalar tensor product 
    """
    arrayTypeAssertion(array)
    output = []
    for item in array:
        if isinstance(item, (list, tuple)):
            output.append(scalarTensorMul(array=item, scalar=scalar))
        else:
            output.append(item * scalar)
    return output
            
    
def shapeSum(*, array):
    """
    Returns the number of elements elements an array of the given shape would hold
    Input: array (iterable)
    Output: sum (number)
    """
    arrayTypeAssertion(array)
    sum = 1
    for el in array:
        sum *= el
    return sum
    
    
def compareElementNumbers(*, a_shape, b_shape):
    """
    Compares the sizes of two different arrays and returns true if they match
    Input: a_shape (iterable / -1), b_shape (iterable / -1)
    Output: boolean
    """
    assert isinstance(a_shape, (list,tuple)) or -1, "shapes should be represented as a list, a tuple or minus 1"
    assert isinstance(b_shape, (list,tuple)) or -1, "shapes should be represented as a list, a tuple or minus 1"
    a_sum = shapeSum(array=a_shape)
    b_sum = shapeSum(array=b_shape)
    return a_sum == b_sum


def flatten(*, array):
    """
    Flattens the array provided
    Input: array (iterable)
    Output: output (iterable)
    """
    arrayTypeAssertion(array)
    if array != []:
        output = []
        for item in array:
            if isinstance(item, (list, tuple)):
                output += flatten(array=item)
            else:
                output.append(item)
        output
    return output

    
def foldArray(*, array, shape):
    """
    Folds(Nests) a flat array into the given shape
    Input: array (iterable), shape (iterable)
    Output: out (iterable)
    """
    length = len(shape)
    for i in range(len(shape)-1):
        out = []
        cur = shape[length - i - 1]
        parts = len(array) / cur
        for j in range(int(parts)):
            portion = array[j*cur:(j*cur)+cur]
            out.append(portion)
        out
    return out

    
def arrayXArraySum(*, array_a, array_b):
    """
    Returns the product for vector sum multiplications
    Input: array_a (iterable), array_b (iterable)
    Output: output (iterable
    """
    arrayTypeAssertion(array_a)
    arrayTypeAssertion(array_b)
    assert len(array_a) == len(array_b), "both arrays should be of equal length"
    output = 0
    for i in range(len(array_a)):
        output += array_a[i] * array_b[i]
    return output


def matrixTransposition(*, matrix):
    """
    Returns the transpose of a matrix
    Input: matrix (iterable)
    Output: output (iterable)
    """
    arrayTypeAssertion(matrix)
    output = createTensor(filler_num=0, shape=(len(matrix[0]), len(matrix)))
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            output[j][i] = matrix[i][j]
    return output
            

    
def reluComp(x):
    """
    Returns input if it is positive and zero otherwise
    Input: x (number)
    Output: (number)
    """
    return (x > 0) * x


def relu(*, matrix):
    """ 
    Applies ReLU activation function elementwise to a matrix
    Input: matrix (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda row: list(map(reluComp, row)),matrix))
    return output

def relu2derivComp(x):
    """
    Returns 1 if input is not 0 and 0 if it is 
    Input: x (number)
    Output: (number)
    """
    return (x >= 0)
    
def relu2deriv(*, matrix):
    """
    Applies ReLU derivative function elementwise to a matrix
    Input: matrix (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda row: list(map(relu2derivComp, row)), matrix))
    return output 

    
def dropOutComp(x):
    """
    Returns 0 or input with a 50% chance of either
    Input: x (number)
    Output: 0|x (number)
    """
    return (random.randint(0,1) * x)


def dropOut(*, matrix):
    """
    Applies dropout function elementwise to a matrix
    Input: matrix (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda row: list(map(dropOutComp, row)), matrix))
    return output


def eleMatrixMultiplication(matrix_a, matrix_b):        
    output = list(map(lambda row1, row2: list(map(lambda el1, el2: el1 * el2, row1, row2)), matrix_a, matrix_b))
    return output

def vectorMatrixMultiplication(*, matrix, vector):
    output = []
    for row in matrix:
        line = vectorElementWiseOperation(tensor_a=row, tensor_b=vector, operator = Operator.MULT)
        output.append(line)
    return output


def matrixMultiplication(*, matrix_a, matrix_b, transpose=[0,0]):
    output = []
    a_shape = getShape(array=matrix_a)
    b_shape = getShape(array=matrix_b)
    assert len(a_shape) == 2 and len(b_shape) == 2 , f"both objects should be 2 dimensional matrices a - {a_shape} b -{b_shape} "
    matrixShapeAssertion(a_shape=a_shape, b_shape=b_shape, transpose=transpose)
    a_x = a_shape[0]
    a_y = a_shape[1]
    b_x = b_shape[0]
    b_y = b_shape[1]
    if transpose[0]:
        for a_i in range(a_y):
            left_hor = [matrix_a[a_j][a_i] for a_j in range(a_x)]
            row = []
            if transpose[1]:
                for b_i in range(b_x):
                    right_ver = matrix_b[b_i]
                    el = sum(vectorMultiplication(vector_a=left_hor, vector_b=right_ver))
                    row.append(el)
            else: 
                for b_j in range(b_y):
                    right_ver = [matrix_b[b_i][b_j] for b_i in range(b_x)]
                    el = sum(vectorMultiplication(vector_a=left_hor, vector_b=right_ver))
                    row.append(el)
            output.append(row)
    else:
        for a_i in range(a_x):
            left_hor = matrix_a[a_i]
            row = []
            if transpose[1]:
                for b_i in range(b_x):
                    right_ver = matrix_b[b_i]
                    el = sum(vectorMultiplication(vector_a=left_hor, vector_b=right_ver))
                    row.append(el)
            else: 
                for b_j in range(b_y):
                    right_ver = [matrix_b[b_i][b_j] for b_i in range(b_x)]
                    el = sum(vectorMultiplication(vector_a=left_hor, vector_b=right_ver))
                    row.append(el)
            output.append(row)
    return output
               
        
def createTensor(*, shape, rand_range=None, filler_num=None, whole=False):
    """
    Creates a tensor of the given shape with random or filler values
    Input: shape (iterable), rand_range (iterable), filler_num (num)
    Output: output (tensor)
    """
    arrayTypeAssertion(shape, "shape")
    assert (rand_range != None) and (filler_num==None) or (filler_num!=None) and (rand_range == None), "Tensor can either be random or populated with filler but not both"
    output = []
    elements = shapeSum(array=shape)
    if filler_num != None:
        # for i in range(len(shape)):
        base = [filler_num for i in range(elements)]
        output = foldArray(array=base, shape=shape)
    elif not whole: 
        base = [((random.random() * (rand_range[1] - rand_range[0]))+rand_range[0]) for i in range(elements)]
        output = foldArray(array=base, shape=shape)
    else:
        base = [random.randint(rand_range[0], rand_range[1])  for i in range(elements)]
        output = foldArray(array=base, shape=shape)
    return output

def weightUpdate(*, layer, layer_delta):
    output = []
    for ele in layer:
        row = [ele * err_val for err_val in layer_delta]
        output.append(row)
    return output
    
def vectorAddition(*, vector_a, vector_b):
    output = list(map(lambda a, b: a + b, vector_a, vector_b))
    return output

def vectorMultiplication(*, vector_a, vector_b):
    va = vector_a
    vb = vector_b
    return [a * b for a, b in zip(va, vb)]
    
def tempDerivWeightMul(*, vector_a, vector_b):
    """
    Temporary function created specifically for the elementwise multiplication of deriv_layer_1 and weights_1_2, do not use elsewhere
    """
    va = vector_a
    vb = vector_b
    return [va[0][i] * vb[i][0] for i in range(len(vb))]

    
def vectorSubtraction(*, vector_a, vector_b):
    """
    Vector elementwise subtraction
    Input: vector_a (iterable), vector (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda a, b: a - b, vector_a, vector_b))
    return output
    

def matrixSubtraction(*, matrix_a, matrix_b):
    """
    Performs elementwise subtraction between the elements of two matrices
    Input: matrix_a (iterable), matrix_b (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda row_a, row_b: list(map(lambda a, b: a - b, row_a, row_b)), matrix_a, matrix_b))
    return output

def matrixAddition(*, matrix_a, matrix_b):
    """
    Performs elementwise subtraction between the elements of two matrices
    Input: matrix_a (iterable), matrix_b (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda row_a, row_b: list(map(lambda a, b: a + b, row_a, row_b)), matrix_a, matrix_b))
    return output

def vectorSquared(*, vector_a):
    """
    Squares the elements of the given vector input
    Input: vector_a (iterable)
    Output: output (iterable)
    """
    output = list(map(lambda a: a**2, vector_a))
    return output
    
    
def vectorElementWiseOperation(*,tensor_a, tensor_b, operator=Operator.ADD):
    """
    Applies a specified operation to the given tensor objects
    Input: tensor_a (iterable), tensor_b (iterable)
    Output: (variable)
    """
    arrayTypeAssertion(tensor_a, "tensor_a")
    arrayTypeAssertion(tensor_b, "tensor_b")
    match operator:
        case Operator.ADD:
           return vectorAddition(vector_a=tensor_a, vector_b=tensor_b)
        case Operator.MINUS:
            return vectorSubtraction(vector_a=tensor_a, vector_b=tensor_b)
        case Operator.MMINUS:
            return matrixSubtraction(matrix_a=tensor_a, matrix_b=tensor_b)
        case Operator.MULT:
            return vectorMultiplication(vector_a=tensor_a, vector_b=tensor_b)
        case _:
            return vectorAddition(vector_a=tensor_a, vector_b=tensor_b)
            
def changeShape(*, array, newShape): 
    """
    Changes the shape of a given tensor to the provided shape
    Input: array (iterable), newShape (iterable)
    Output: final (tensor) / array
    
    """
    shape = getShape(array=array)
    assert compareElementNumbers(a_shape=shape, b_shape=newShape) or newShape == -1, f"cannot reshape array of shape {shape} into shape {newShape}"
    if newShape == -1:
        return flatten(aray=array)
    else:
        flat = flatten(array=array)
        final = foldArray(array=flat, shape=newShape)
    return final     

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Data loaded successfully!!")
(x_train, y_train), (x_test, y_test) = (x_train.tolist(), y_train.tolist()), (x_test.tolist(), y_test.tolist())

random.seed(1)

images, labels = (changeShape(array = x_train[0:1000], newShape=(1000, 28*28)), y_train[0:1000])
images = scalarTensorDiv(array=images, scalar=255)
one_hot_labels = createTensor(shape=(1000,10), filler_num=0)

test_length = len(x_test)
test_images, test_labels = (changeShape(array=x_test, newShape=(test_length, 28*28)), y_test)
test_images = scalarTensorDiv(array=test_images, scalar=255) 
test_one_hot_labels = createTensor(shape=(test_length, 10), filler_num=0)

for i, j in enumerate(labels):
    one_hot_labels[i][j] = 1

for i, j in enumerate(test_labels):
    test_one_hot_labels[i][j] = 1
    
labels = one_hot_labels
test_labels = test_one_hot_labels
alpha, iterations = (0.005, 300)
pixels_per_image, num_labels, hidden_size = (784, 10, 100)

layer_1_shape = (1,hidden_size)

weights_0_1 = createTensor(shape=(pixels_per_image, hidden_size), rand_range=(-0.1,0.1))
weights_1_2 = createTensor(shape=(hidden_size, num_labels), rand_range=(-0.1, 0.1))

print("Random Weight values initialized!!!")

Data loaded successfully!!
Random Weight values initialized!!!


In [3]:
for i in range(iterations):
    correct_cnt = 0
    test_cnt = 0
    for j in range(len(images)):
        print(f"{j}/{len(images)}", end="\r")
        layer_0 = images[j:j+1]
        layer_1 = matrixMultiplication(matrix_a=layer_0, matrix_b=weights_0_1)

        #apply non linear function
        layer_1 = relu(matrix=layer_1)

        

        #apply drop out mask to reduce number of effective weights
        #also avoids overdependency on a single set of weights
        #ensures all weights are playing a role
        dropout_mask = scalarTensorMul(array=createTensor(shape=layer_1_shape, rand_range=(0, 1), whole=True), scalar=2)

        layer_1 = eleMatrixMultiplication(matrix_a=layer_1, matrix_b=dropout_mask)

        # layer 1 dot weights_1_2 for output
        layer_2 = matrixMultiplication(matrix_a=layer_1, matrix_b=weights_1_2)



        
        #keeps score
        if layer_2[0].index(max(layer_2[0])) == labels[j].index(max(labels[j])):
            correct_cnt += 1

        
        #layer 2 delta is elementWiseSubtraction of n_answer(layer 2) from correct_answer(label_row)
        error = vectorElementWiseOperation(tensor_a=layer_2, tensor_b=labels[j:j+1], operator=Operator.MMINUS)

        error_delta = scalarTensorMul(array=error, scalar=alpha)

        #weight layer delta with your input layer 1 
        #this reincorporates dropout into the delta (error) calculation
        layer_2_delta = matrixMultiplication(matrix_a=layer_1, matrix_b=error_delta, transpose=[1,0])



        #weight error with the previous layers weights
        layer_1_delta = matrixMultiplication(matrix_a=error_delta, matrix_b=weights_1_2, transpose=[0,1])



        #change weights_1_2 based on new weighted layer_delta value
        #a negative weight layer value indicates that the label value was greater than the predicted value
        #subracting against a negative value is addition increasing weights to meet higher labels and vice versa

        weights_1_2 = matrixSubtraction(matrix_a=weights_1_2, matrix_b=layer_2_delta,)

        #relu prevented some weights from playing 
        #use relu2deriv to prevent those weights from changing due to outcomes they didn't contibute to

        deriv_layer_1 = relu2deriv(matrix=layer_1)


        layer_1_delta = vectorMultiplication(vector_a=deriv_layer_1[0], vector_b=layer_1_delta)


        


        #add in alpha values to moderate change by arbitrary but informed number 
        layer_1_delta = eleMatrixMultiplication(matrix_a=layer_1_delta, matrix_b=dropout_mask)


 
        weight_update = matrixMultiplication(matrix_a=layer_0, matrix_b=layer_1_delta, transpose=[1,0])
        #change weight value
        weights_0_1 = matrixSubtraction(matrix_a=weights_0_1, matrix_b=weight_update)


        

    # for k in range(test_length):
        
    #     layer_0 = test_images[k:k+1]
    #     layer_1 = matrixMultiplication(matrix_a=layer_0, matrix_b=weights_0_1)
    #     layer_1 = relu(matrix=layer_1)
    #     layer_2 = matrixMultiplication(matrix_a=layer_1, matrix_b= weights_1_2)
    #     if layer_2[0].index(max(layer_2[0])) == test_labels[k].index((max(test_labels[k]))):
    #         test_cnt += 1

    # + " Test-Acc:" + str(test_cnt/float(len(test_images))) 
    # print("\n" + str(correct_cnt))
    sys.stdout.write("I:" + str(i) + " Train-Acc:" + str(correct_cnt/float(len(images))) + "\n")
        

I:0 Train-Acc:0.446
I:1 Train-Acc:0.68
I:2 Train-Acc:0.705
I:3 Train-Acc:0.728
I:4 Train-Acc:0.748
I:5 Train-Acc:0.771
I:6 Train-Acc:0.766
I:7 Train-Acc:0.771
I:8 Train-Acc:0.771
I:9 Train-Acc:0.779
I:10 Train-Acc:0.765
I:11 Train-Acc:0.791
I:12 Train-Acc:0.799
I:13 Train-Acc:0.801
I:14 Train-Acc:0.812
I:15 Train-Acc:0.821
I:16 Train-Acc:0.815
I:17 Train-Acc:0.812
I:18 Train-Acc:0.819
I:19 Train-Acc:0.823
I:20 Train-Acc:0.822
I:21 Train-Acc:0.813
I:22 Train-Acc:0.829
I:23 Train-Acc:0.827
I:24 Train-Acc:0.826
I:25 Train-Acc:0.825
I:26 Train-Acc:0.831
I:27 Train-Acc:0.842
I:28 Train-Acc:0.834
I:29 Train-Acc:0.86
I:30 Train-Acc:0.833
I:31 Train-Acc:0.834
I:32 Train-Acc:0.838
I:33 Train-Acc:0.859
I:34 Train-Acc:0.861
I:35 Train-Acc:0.856
I:36 Train-Acc:0.844
I:37 Train-Acc:0.861
I:38 Train-Acc:0.848
I:39 Train-Acc:0.856
I:40 Train-Acc:0.839
I:41 Train-Acc:0.854
I:42 Train-Acc:0.848
I:43 Train-Acc:0.842
I:44 Train-Acc:0.859
I:45 Train-Acc:0.853
I:46 Train-Acc:0.845
I:47 Train-Acc:0.865
I:48